# Gamble Battle Endless Economy Model

## tl;dr

This notebook models the economy from the live July 16, 2026 checkout. It tests
probability-based payouts, fixed chapter-derived prices, several betting
policies, and the corrected sticky Stakes-market model that preserves meaningful
unit purchases under exponential bankroll growth. Conclusions are populated by
the executed cells below; the blueprint report is the decision-facing artifact.

## Context & Methods

The design target is a multi-session run where a strong normal player usually
enters fights at 60–75% projected win odds, wagers a minority of bankroll, and
uses all-ins only for exceptional reads.

### Key Assumptions

- Contract prices use the fixed reference curve.
- Unit, reroll, and XP prices use a visible, irreversible Stakes denomination
  that follows the normal depth schedule but may promote early from the run’s
  high-water bankroll.
- Stakes promotions occur only between chapters and never reverse after spending.
- The saved odds calibration is directionally useful but too small to authorize
  production payouts by itself.
- The model abstracts qualitative contracts as the power increment required to
  keep pace with the independent chapter curve.
- Current unit-cost balance is compared with the live 1.5× stat multiplier.
- Real player telemetry does not yet exist, so recommendations remain a design
  baseline for playtesting rather than a final shipped balance.

In [1]:
from pathlib import Path
import csv
import json
import math

import economy_model
import unit_market_model

ROOT = Path.cwd()
baseline = economy_model.load_baseline()
result = economy_model.run_model(simulations=5000, chapters=40)
unit_market_result = unit_market_model.build_results()
result["selected_config"]

{'target_probability': 0.68,
 'target_net_growth': 1.22,
 'payout_bonus': 0.45,
 'contract_share': 0.2,
 'simulation_chapters': 40,
 'curve_horizon_chapters': 80,
 'simulations_per_policy': 5000}

## Data

In [2]:
baseline_summary = {
    "playable_units": baseline["shop"]["total_playable_units"],
    "cost_distribution": baseline["shop"]["unit_count_by_cost"],
    "max_player_level": baseline["shop"]["maximum_player_level"],
    "max_board_capacity": baseline["shop"]["maximum_board_capacity"],
    "copies_for_level_4": baseline["units"]["copies_for_level_4"],
    "gold_to_max_level": baseline["shop"]["gold_to_max_level_at_current_rate"],
    "full_level_4_five_cost_board": baseline["units"]["sixteen_level_4_five_cost_units_base_spend"],
    "odds_calibration_samples": baseline["odds_estimator"]["calibration_samples"],
}
baseline_summary

{'playable_units': 51,
 'cost_distribution': {'1': 14, '2': 13, '3': 12, '4': 8, '5': 4},
 'max_player_level': 14,
 'max_board_capacity': 16,
 'copies_for_level_4': 27,
 'gold_to_max_level': 2224,
 'full_level_4_five_cost_board': 2160,
 'odds_calibration_samples': 144}

## Results

In [3]:
result["unit_price_candidates"]

[{'candidate': 'current_1_2_3_4_5',
  'prices': [1, 2, 3, 4, 5],
  'price_per_power': [1.0, 1.3333, 1.3333, 1.1852, 0.9877],
  'price_per_power_cv': 0.1303,
  'top_to_bottom_price_ratio': 5.0,
  'fixed_ladder_extra_doublings_vs_current': 0.0},
 {'candidate': 'moderate_1_2_4_7_11',
  'prices': [1, 2, 4, 7, 11],
  'price_per_power': [1.0, 1.3333, 1.7778, 2.0741, 2.1728],
  'price_per_power_cv': 0.2662,
  'top_to_bottom_price_ratio': 11.0,
  'fixed_ladder_extra_doublings_vs_current': 1.138},
 {'candidate': 'broad_1_3_10_30_100',
  'prices': [1, 3, 10, 30, 100],
  'price_per_power': [1.0, 2.0, 4.4444, 8.8889, 19.7531],
  'price_per_power_cv': 0.9469,
  'top_to_bottom_price_ratio': 100.0,
  'fixed_ladder_extra_doublings_vs_current': 4.322}]

In [4]:
{
    "market_rule": unit_market_result["market_rule"],
    "price_rule": unit_market_result["price_rule"],
    "million_gold_scenario": next(
        row for row in unit_market_result["scenarios"]
        if row["peak_bankroll"] == 1_000_000
    ),
    "greedy_stress": unit_market_result["greedy_five_shop_stress"],
}

{'market_rule': 'U is the largest 1-2-5 denomination not greater than peak_bankroll / 50; promotion is irreversible and occurs at chapter boundaries.',
 'price_rule': 'unit price = rarity tier * U; reroll = 2U; XP or command = 4U',
 'million_gold_scenario': {'peak_bankroll': 1000000,
  'market_unit': 20000,
  'effective_market_reserve_units': 50.0,
  'one_cost_price': 20000,
  'two_cost_price': 40000,
  'three_cost_price': 60000,
  'four_cost_price': 80000,
  'five_cost_price': 100000,
  'reroll_price': 40000,
  'xp_or_command_price': 80000,
  'expected_level_14_shop_price': 396000.0,
  'five_cost_share_of_peak': 0.1,
  'reroll_share_of_peak': 0.04,
  'expected_shop_share_of_peak': 0.396},
 'greedy_stress': {'simulations': 20000,
  'shops_per_run': 5,
  'mean_offer_buy_rate': 0.6909595000000001,
  'mean_full_shop_rate': 0.40019,
  'median_remaining_bankroll_share': 0.0}}

In [5]:
unit_market_result["specific_unit_search_level_14"]

[{'level': 14,
  'tier': 1,
  'identities_in_tier': 14,
  'specific_unit_slot_probability': 0.002142857142857143,
  'specific_unit_shop_probability': 0.010668465638060454,
  'expected_rerolls': 92.73419139416208,
  'expected_search_and_purchase_units': 186.46838278832416,
  'share_of_50_unit_reserve': 3.729367655766483},
 {'level': 14,
  'tier': 2,
  'identities_in_tier': 13,
  'specific_unit_slot_probability': 0.004615384615384615,
  'specific_unit_shop_probability': 0.022864886217562774,
  'expected_rerolls': 42.73518374352936,
  'expected_search_and_purchase_units': 87.47036748705872,
  'share_of_50_unit_reserve': 1.7494073497411744},
 {'level': 14,
  'tier': 3,
  'identities_in_tier': 12,
  'specific_unit_slot_probability': 0.015,
  'specific_unit_shop_probability': 0.07278349763437508,
  'expected_rerolls': 12.739378190142208,
  'expected_search_and_purchase_units': 28.478756380284416,
  'share_of_50_unit_reserve': 0.5695751276056883},
 {'level': 14,
  'tier': 4,
  'identities_in_

In [6]:
policy_rows = [
    row for row in result["policy_summary"]
    if not row["policy"].startswith("_")
]
policy_rows

[{'policy': 'odds_aware',
  'simulations': 5000,
  'survival_rate': 0.7018,
  'median_chapters_completed': 40.0,
  'p10_bankroll': 0.41,
  'median_bankroll': 3523.62,
  'p90_bankroll': 2472855348.96,
  'median_total_earned': 31809.88,
  'median_contracts_bought': 28.0,
  'mean_win_probability': 0.6701,
  'mean_bet_fraction': 0.2917,
  'mean_all_in_rate': 0.0083,
  'target_chapters': 40},
 {'policy': 'fixed_25',
  'simulations': 5000,
  'survival_rate': 0.7024,
  'median_chapters_completed': 40.0,
  'p10_bankroll': 0.35,
  'median_bankroll': 1396.54,
  'p90_bankroll': 170401732.03,
  'median_total_earned': 15162.99,
  'median_contracts_bought': 21.0,
  'mean_win_probability': 0.6635,
  'mean_bet_fraction': 0.2691,
  'mean_all_in_rate': 0.0085,
  'target_chapters': 40},
 {'policy': 'conservative',
  'simulations': 5000,
  'survival_rate': 0.8166,
  'median_chapters_completed': 40.0,
  'p10_bankroll': 0.72,
  'median_bankroll': 1081.82,
  'p90_bankroll': 29739.63,
  'median_total_earned':

In [7]:
with open("recommended_curve.csv", newline="", encoding="utf-8") as handle:
    curve_rows = list(csv.DictReader(handle))

selected_chapters = {1, 5, 10, 20, 30, 40}
[
    row for row in curve_rows
    if int(row["chapter"]) in selected_chapters
]

[{'chapter': '1',
  'reference_bankroll': '3.0',
  'enemy_base_rating': '100.0',
  'average_enemy_rating': '75.0',
  'required_player_rating_for_68pct': '147.01',
  'incremental_power_required': '147.01',
  'standard_contract_price': '1',
  'bold_contract_price': '2',
  'reroll_price': '2',
  'xp_or_command_price': '4',
  'donor_contract_fee': '5',
  'target_probability': '0.68',
  'gross_payout_at_target_odds': '2.1324',
  'payout_bonus': '0.45'},
 {'chapter': '5',
  'reference_bankroll': '6.65',
  'enemy_base_rating': '172.0',
  'average_enemy_rating': '129.0',
  'required_player_rating_for_68pct': '252.86',
  'incremental_power_required': '26.46',
  'standard_contract_price': '1',
  'bold_contract_price': '2',
  'reroll_price': '2',
  'xp_or_command_price': '4',
  'donor_contract_fee': '5',
  'target_probability': '0.68',
  'gross_payout_at_target_odds': '2.1324',
  'payout_bonus': '0.45'},
 {'chapter': '10',
  'reference_bankroll': '17.96',
  'enemy_base_rating': '297.0',
  'averag

In [8]:
odds_exponent = baseline["odds_estimator"]["odds_exponent"]
{
    probability: round(
        economy_model.rating_ratio_for_probability(probability, odds_exponent),
        3,
    )
    for probability in (0.60, 0.68, 0.75)
}

{0.6: 1.436, 0.68: 1.96, 0.75: 2.667}

## Takeaways

The final report should treat the following as the model’s decision tests:

1. Literal 1–5 gold forever is rejected. Preserve the 1–5 ratio through a
   visible Stakes unit `U`.
2. Use `unit price = rarity × U`, `reroll = 2U`, and `XP/command = 4U`.
3. Promote `U` irreversibly through a 1-2-5 ladder at chapter boundaries, using
   the depth schedule normally and high-water wealth to prevent rich outliers
   from permanently trivializing shops.
4. Higher-Stakes shops must sell current-depth or premium recruits; repricing
   the same obsolete unit is fake inflation.
5. Units and the next wager must compete for post-shopping liquid gold so buying
   a unit damages the player’s compounding engine.
6. Probability-based payouts can make all-in play mathematically positive but
   strategically overbet, allowing large-money runs without making all-ins the
   default.
7. The “hold ten times the next price” policy must not outperform the odds-aware
   policy on both survival and total earned.
8. The selected coefficients require broader in-engine odds calibration and
   real playtest telemetry before implementation.